In [5]:
import openai
import pandas as pd
import chromadb
import os
import shutil
from openai import OpenAI
from chromadb.config import Settings
from dotenv import load_dotenv
from docx import Document
import tiktoken  # For token counting (specific to OpenAI models)

In [6]:
# Load openAI api key
load_dotenv()

# Access the API key
api_key = os.getenv("API_KEY")

In [ ]:
# Example Usage
file_path = "D:\Project\Chatbot\Company Data.docx"
sections = load_docx(file_path)
chunks = chunk_by_tokens(sections, max_tokens=512)

# Print or process the chunks
for chunk in chunks:
    print(f"Heading: {chunk['heading']}")
    print(f"Content: {chunk['content']}")
    print("-" * 50)

In [ ]:
## Embedding company data to store in ChromaDB

# Set OpenAI API key
client = OpenAI(api_key=api_key)

# Step 1: Load Excel file into a pandas DataFrame
excel_file_path = "D:\Project\Chatbot\company_data.xlsx"
df = pd.read_excel(excel_file_path)

# Step 2: Generate embeddings using OpenAI API
def generate_embeddings(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",  # Or the appropriate embedding model
        input=text
    )
    return response.data[0].embedding  # Correct way to access the embedding

# Step 3: Initialize Persistent ChromaDB client
persist_directory = "D:/Project/Chatbot/chromadb_data"
if os.path.exists(persist_directory):
    shutil.rmtree(persist_directory)

os.makedirs(persist_directory)
persistentClient = chromadb.PersistentClient(path=persist_directory)

# Step 4: Create or retrieve Chroma collection
collection_name = "website_embeddings"
if collection_name in [col.name for col in persistentClient.list_collections()]:
    collection = persistentClient.get_collection(collection_name)
else:
    collection = persistentClient.create_collection(collection_name)

# Step 5: Add embeddings to ChromaDB
for index, row in df.iterrows():
    # Combine Title, Description, and URL for embeddings
    combined_text = f"{row['Title']}: {row['Description']} (URL: {row['URL/Link']})"
    
    # Generate embedding for the combined text
    embedding = generate_embeddings(combined_text)
    
    # Store embedding and metadata in ChromaDB
    collection.add(
        documents=[combined_text],
        embeddings=[embedding],
        metadatas=[{
            "Page/Section": row['Page/Section'],
            "Category": row['Category'],
            "Title": row['Title'],
            "Description": row['Description'],
            "URL/Link": row['URL/Link']
        }],
        ids=[str(index)]  # Use row index or any unique identifier
    )
    print(f"Processed row {index + 1}/{len(df)}")

print("Embeddings with URLs have been added to ChromaDB.")

In [ ]:
# Retrieve the collection
persist_directory = "D:/Project/Chatbot/chromadb_data"
persistentClient = chromadb.PersistentClient(path=persist_directory)
retrieved_collection = persistentClient.get_collection(name="website_embeddings")

# Fetch all documents, embeddings, and metadata
all_data = retrieved_collection.get(include=["documents", "metadatas", "embeddings"])

# Print all retrieved information
for i, document in enumerate(all_data["documents"]):
    print(f"\nDocument {i + 1}:")
    print("Document Content:", document)
    print("Metadata:", all_data["metadatas"][i])
    print("Embedding (first 5 values):", all_data["embeddings"][i][:5])  # Only print first 5 values for brevity

In [8]:
# Initialize the Persistent ChromaDB client (without path argument)
client_chroma = chromadb.PersistentClient()

# Retrieve the collection
collection_name = "website_embeddings"
retrieved_collection = persistentClient.get_collection(name=collection_name)

# Function to query ChromaDB for relevant documents
def query_chromadb(query, collection):
    # Generate the embedding for the query using OpenAI API
    query_embedding = client.embeddings.create(
        model="text-embedding-3-small",  # Embedding model
        input=query
    ).data[0].embedding
    
    # Query the collection for the closest documents based on cosine similarity
    results = collection.query(
        query_embeddings=[query_embedding], 
        n_results=3  # Number of results to retrieve
    )
    
    # Debug: Print the raw structure of the results to understand its structure
    print("Query Results Structure:", results)

    # Extract documents and metadatas from the list of results
    # Assuming results is a list of dictionaries where each dictionary contains the relevant info
    documents = []
    metadatas = []

    # Check if results is a list and iterate through it to collect documents and metadata
    if isinstance(results, list):
        for result in results:
            documents.append(result.get('document', 'No document found'))
            metadatas.append(result.get('metadata', {}))  # Default to empty dict if no metadata found
    
    return documents, metadatas

# Function to interact with GPT-4 and generate a response
def generate_response(user_query):
    # Step 1: Query ChromaDB for relevant documents
    documents, metadatas = query_chromadb(user_query, retrieved_collection)
    
    # Step 2: Format the documents and metadata for GPT-4
    context = "\n".join([f"Title: {meta['Title']}\nDescription: {meta['Description']}\nURL: {meta['URL/Link']}\n" 
                         for meta in metadatas])
    
    # Step 3: Generate the GPT-4 response using the retrieved context
    response = openai.chat.completions.create(  
        model="gpt-4o-mini",  # Specify the GPT-4o-mini model
        messages=[ 
            {"role": "system", "content": "You are a helpful assistant to help user answer the question about iCONEXT Company"},
            {"role": "user", "content": f"{user_query}\n\nContext:\n{context}\n help to answer the question by using the context and your knowledge"}
        ]
    )
    
    # Accessing the content from the response object correctly
    return response.choices[0].message.content


In [9]:
# Example 
client = OpenAI(api_key=api_key)
user_query = "What is Davinci Labs"
response = generate_response(user_query)
print(response)

Query Results Structure: {'ids': [['5', '12', '6']], 'embeddings': None, 'documents': [['Davinci Labs, Davinci lab: Davinci Labs automate and simplify machine learning technology and enable data analysis with just a few clicks.\n\nDavinci Labs is AI (Artificial Intelligence) based data analytics system. Until now Machine Learning technology has been occupied by only a few experts.\n\nIntuitive user interface that anyone can easily start analyzing by machine learning\n\nNo coding required, just click to adjust parameters of algorithm, you can create a variety of models.\n\n\nDavinci Labs\nAutomated Machine Learning (Auto ML)  is the process of automating the process of applying machine learning to real-world problems. Auto ML covers the complete pipeline from the raw dataset to the deployable machine learning model. Auto ML was proposed as an artificial intelligence-based solution to the ever-growing challenge of applying machine learning. The high degree of automation in Auto ML allows

In [6]:
client = OpenAI(api_key=api_key)
user_query = "What is Davinci Labs"
query_chromadb(user_query, retrieved_collection)

Query Results Structure: {'ids': [['5', '12', '6']], 'embeddings': None, 'documents': [['Davinci Labs, Davinci lab: Davinci Labs automate and simplify machine learning technology and enable data analysis with just a few clicks.\n\nDavinci Labs is AI (Artificial Intelligence) based data analytics system. Until now Machine Learning technology has been occupied by only a few experts.\n\nIntuitive user interface that anyone can easily start analyzing by machine learning\n\nNo coding required, just click to adjust parameters of algorithm, you can create a variety of models.\n\n\nDavinci Labs\nAutomated Machine Learning (Auto ML)  is the process of automating the process of applying machine learning to real-world problems. Auto ML covers the complete pipeline from the raw dataset to the deployable machine learning model. Auto ML was proposed as an artificial intelligence-based solution to the ever-growing challenge of applying machine learning. The high degree of automation in Auto ML allows

([], [])